# Feature Enrichment Pipeline

This notebook is the second step of the cleaned HDB resale pipeline:

`01_data_preprocessing_pipeline.ipynb` -> `02_feature_enrichment_pipeline.ipynb`

What this notebook does:

- loads the intermediate output from `01_data_preprocessing_pipeline.ipynb`
- derives shared housing attributes such as `region`, `region_URA`, `mature_estate`, and `remaining_lease_years`
- backfills locally cached geospatial and MRT features from files stored in `data/`
- computes the enrichments that can be derived locally
- filters the pipeline output to the project window from `2015-01` through `2025-12`
- saves the official final modeling dataset

Final output:

- `data/hdb_resale_final_modeling_dataset_2015_2025.csv`


In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()
INTERMEDIATE_PATH = PROJECT_ROOT / "data" / "hdb_resale_pipeline_intermediate.csv"
RAW_PATH = PROJECT_ROOT / "data" / "Resale Flat Prices from Jan 2015 to Feb 2026.csv"
CACHED_FEATURE_CANDIDATES = [
    PROJECT_ROOT / "data" / "hdb_resale_with_new_features.csv",
    PROJECT_ROOT / "data" / "hdb_resale_with_walking_distance_FINAL.csv",
    PROJECT_ROOT / "data" / "merged_resale_dataset.csv",
]
GDP_PATH = PROJECT_ROOT / "data" / "gdp_per_capita_cleaned.csv"
SORA_PATH = PROJECT_ROOT / "data" / "Domestic_Interest_Rates.csv"
PARK_GEOJSON_PATH = PROJECT_ROOT / "data" / "NParksParksandNatureReserves.geojson"
HAWKER_GEOJSON_PATH = PROJECT_ROOT / "data" / "HawkerCentresGEOJSON.geojson"
FINAL_OUTPUT_PATH = PROJECT_ROOT / "data" / "hdb_resale_final_modeling_dataset_2015_2025.csv"
PIPELINE_START_DATE = pd.Timestamp("2015-01-01")
PIPELINE_END_DATE = pd.Timestamp("2025-12-01")

FINAL_MODEL_COLUMNS = [
    "town",              # Retained for Target Encoding / Microscopic Inference
    "region",
    "region_URA",
    "mature_estate",
    "flat_type",         
    "flat_model_raw",    
    "flat_model",        
    "floor_area_sqm",
    "storey_mid",
    "remaining_lease_years",
    "mrt_walking_distance",
    "dist_to_cbd_m",
    "dist_nearest_park_m",
    "dist_nearest_hawker_centre_m",
    "sora_monthly_avg",
    "Per_Capita_GDP",
    "days_since_start",  
    "resale_price",
    "log_resale_price",
]

In [3]:
def parse_remaining_lease_years(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text:
        return np.nan
    try:
        return float(text)
    except ValueError:
        pass
    match = re.match(r"(\d+)\s*years?(?:\s*(\d+)\s*months?)?", text, flags=re.IGNORECASE)
    if not match:
        return np.nan
    years = int(match.group(1))
    months = int(match.group(2)) if match.group(2) else 0
    return years + months / 12.0

def parse_storey_bounds(value):
    match = re.match(r"\s*(\d+)\s*TO\s*(\d+)\s*", str(value), flags=re.IGNORECASE)
    if not match:
        return np.nan, np.nan, np.nan
    lower = int(match.group(1))
    upper = int(match.group(2))
    midpoint = (lower + upper) / 2.0
    return lower, upper, midpoint

def haversine_distance(lat1, lon1, lat2, lon2):
    radius_km = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return radius_km * c

MATURE_ESTATES = {
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT MERAH", "BUKIT TIMAH",
    "CENTRAL AREA", "CLEMENTI", "GEYLANG", "KALLANG/WHAMPOA",
    "MARINE PARADE", "PASIR RIS", "QUEENSTOWN", "SERANGOON",
    "TAMPINES", "TOA PAYOH"
}

CCR_TOWNS = {"BUKIT TIMAH", "CENTRAL AREA"}
RCR_TOWNS = {"BISHAN", "BUKIT MERAH", "KALLANG/WHAMPOA", "MARINE PARADE", "QUEENSTOWN", "GEYLANG", "TOA PAYOH"}

URA_REGION_MAP = {
    "Central": ["BISHAN", "BUKIT MERAH", "BUKIT TIMAH", "CENTRAL AREA", "GEYLANG", "KALLANG/WHAMPOA",
                "MARINE PARADE", "QUEENSTOWN", "TOA PAYOH"],
    "North": ["SEMBAWANG", "WOODLANDS", "YISHUN"],
    "East": ["BEDOK", "PASIR RIS", "TAMPINES"],
    "West": ["BUKIT BATOK", "BUKIT PANJANG", "CHOA CHU KANG", "CLEMENTI", "JURONG EAST", "JURONG WEST"],
    "North-East": ["ANG MO KIO", "HOUGANG", "PUNGGOL", "SENGKANG", "SERANGOON"],
}

TOWN_TO_URA_REGION = {
    town: region
    for region, towns in URA_REGION_MAP.items()
    for town in towns
}

def classify_region(town):
    town = str(town).strip().upper()
    if town in CCR_TOWNS:
        return "CCR"
    if town in RCR_TOWNS:
        return "RCR"
    return "OCR"

def classify_ura_region(town):
    town = str(town).strip().upper()
    return TOWN_TO_URA_REGION.get(town, np.nan)

MATCH_COLS = [
    "month", "town", "flat_type", "block", "street_name", "storey_range_key",
    "floor_area_sqm", "lease_commence_date", "remaining_lease", "resale_price"
]

BACKFILL_COLS = [
    "LATITUDE", "LONGITUDE", "mrt_name", "mrt_straight_distance", "mrt_walking_distance",
    "dist_to_cbd_m", "dist_nearest_park_m", "dist_nearest_hawker_centre_m",
    "sora_monthly_avg", "mature_estate", "region", "flat_type_encoded",
    "month_date", "sale_year", "sale_month", "month_period",
    "remaining_lease_years", "remaining_lease_months", "remaining_lease_float",
    "storey_min", "storey_max", "storey_mid", "flat_age_years", "Per_Capita_GDP"
]

REQUIRED_SNAPSHOT_COLS = {
    "month", "town", "flat_type", "block", "street_name", "storey_range",
    "floor_area_sqm", "lease_commence_date", "remaining_lease", "resale_price",
    "sale_year", "sale_month", "month_date", "month_period",
    *BACKFILL_COLS,
}

STRING_KEY_COLS = ["month", "town", "flat_type", "block", "street_name", "storey_range_key"]
NUMERIC_KEY_COLS = ["floor_area_sqm", "lease_commence_date", "resale_price"]

def normalize_string_key(series):
    return series.astype("string").str.strip().fillna("<NA>").str.upper()

def normalize_numeric_key(series):
    numeric = pd.to_numeric(series, errors="coerce")
    def format_value(value):
        if pd.isna(value):
            return "<NA>"
        value = float(value)
        if value.is_integer():
            return str(int(value))
        return (f"{value:.6f}").rstrip("0").rstrip(".")
    return numeric.map(format_value)

def normalize_remaining_lease_key(series):
    parsed = series.apply(parse_remaining_lease_years)
    return normalize_numeric_key(parsed)

def prepare_match_frame(frame):
    temp = frame.copy()
    if "storey_range_raw" in temp.columns:
        temp["storey_range_key"] = temp["storey_range_raw"]
    elif "storey_range" in temp.columns:
        temp["storey_range_key"] = temp["storey_range"]
    else:
        temp["storey_range_key"] = pd.Series("", index=temp.index, dtype="object")
    for col in STRING_KEY_COLS:
        temp[col] = normalize_string_key(temp[col])
    for col in NUMERIC_KEY_COLS:
        temp[col] = normalize_numeric_key(temp[col])
    temp["remaining_lease"] = normalize_remaining_lease_key(temp["remaining_lease"])
    temp["__match_occurrence"] = temp.groupby(MATCH_COLS, dropna=False).cumcount()
    return temp

def load_snapshot(path):
    snap = pd.read_csv(
        path,
        low_memory=False,
        usecols=lambda c: (not str(c).startswith("Unnamed:")) and (c in REQUIRED_SNAPSHOT_COLS or c == "Year"),
    )
    if "Year" in snap.columns and "sale_year" not in snap.columns:
        snap = snap.rename(columns={"Year": "sale_year"})
    if "month_date" in snap.columns:
        snap["month_date"] = pd.to_datetime(snap["month_date"], errors="coerce")
    elif "month" in snap.columns:
        snap["month_date"] = pd.to_datetime(snap["month"] + "-01", errors="coerce")
    if "sale_year" not in snap.columns and "month_date" in snap.columns:
        snap["sale_year"] = snap["month_date"].dt.year
    if "sale_month" not in snap.columns and "month_date" in snap.columns:
        snap["sale_month"] = snap["month_date"].dt.month
    if "month_period" not in snap.columns and "month_date" in snap.columns:
        snap["month_period"] = snap["month_date"].dt.to_period("M").astype(str)
    if "remaining_lease_years" not in snap.columns and "remaining_lease" in snap.columns:
        snap["remaining_lease_years"] = snap["remaining_lease"].apply(parse_remaining_lease_years)
    if "remaining_lease_months" not in snap.columns and "remaining_lease_years" in snap.columns:
        snap["remaining_lease_months"] = (snap["remaining_lease_years"] * 12).round()
    if "remaining_lease_float" not in snap.columns and "remaining_lease_years" in snap.columns:
        snap["remaining_lease_float"] = snap["remaining_lease_years"]
    if "flat_type_encoded" not in snap.columns and "flat_type" in snap.columns:
        flat_type_order = {
            "1 ROOM": 1,
            "2 ROOM": 2,
            "3 ROOM": 3,
            "4 ROOM": 4,
            "5 ROOM": 5,
            "EXECUTIVE": 6,
            "MULTI-GENERATION": 7,
        }
        snap["flat_type_encoded"] = snap["flat_type"].astype(str).str.strip().str.upper().map(flat_type_order)
    return snap

def build_final_dataset(frame):
    missing_columns = [column for column in FINAL_MODEL_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise ValueError("Missing required final columns: " + ", ".join(missing_columns))
    return frame.loc[:, FINAL_MODEL_COLUMNS].copy()

## Load Inputs And Validate Alignment

In [4]:
if not INTERMEDIATE_PATH.exists():
    raise FileNotFoundError(f"Missing intermediate dataset: {INTERMEDIATE_PATH}")
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Missing raw resale dataset: {RAW_PATH}")

df = pd.read_csv(INTERMEDIATE_PATH, low_memory=False)
raw_reference = pd.read_csv(RAW_PATH, low_memory=False)

print(f"Intermediate input: {INTERMEDIATE_PATH}")
print(f"Raw reference: {RAW_PATH}")
print(f"Intermediate rows: {len(df):,}")
print(f"Raw reference rows: {len(raw_reference):,}")

key_columns = ["month", "town", "flat_type", "floor_area_sqm", "lease_commence_date", "remaining_lease", "resale_price"]
if len(df) != len(raw_reference):
    raise ValueError("Intermediate dataset row count does not match the raw reference file.")
for column in key_columns:
    left = df[column].reset_index(drop=True).astype(str)
    right = raw_reference[column].reset_index(drop=True).astype(str)
    if not left.equals(right):
        raise ValueError(f"Intermediate dataset no longer aligns row-for-row with raw reference on column: {column}")

Intermediate input: C:\Users\Study\OneDrive - National University of Singapore\Desktop\Y3S2\CS3244\CS3244_group-3_ResalePriceModeling\data\hdb_resale_pipeline_intermediate.csv
Raw reference: C:\Users\Study\OneDrive - National University of Singapore\Desktop\Y3S2\CS3244\CS3244_group-3_ResalePriceModeling\data\Resale Flat Prices from Jan 2015 to Feb 2026.csv
Intermediate rows: 262,855
Raw reference rows: 262,855


## Derive Shared Attributes Needed Downstream

In [5]:
raw_storey = raw_reference[["storey_range"]].copy().rename(columns={"storey_range": "storey_range_raw"})
bounds = raw_storey["storey_range_raw"].apply(parse_storey_bounds)
raw_storey[["storey_min", "storey_max", "storey_mid_from_raw"]] = pd.DataFrame(bounds.tolist(), index=raw_storey.index)

if "block" not in df.columns:
    df["block"] = raw_reference["block"]
else:
    df["block"] = df["block"].fillna(raw_reference["block"])

if "street_name" not in df.columns:
    df["street_name"] = raw_reference["street_name"]
else:
    df["street_name"] = df["street_name"].fillna(raw_reference["street_name"])

df["storey_range_raw"] = raw_storey["storey_range_raw"]
df["storey_min"] = raw_storey["storey_min"]
df["storey_max"] = raw_storey["storey_max"]
df["storey_mid"] = pd.to_numeric(df["storey_mid"], errors="coerce").fillna(raw_storey["storey_mid_from_raw"])

df["month_date"] = pd.to_datetime(df["month"] + "-01", errors="coerce")
df["sale_year"] = df["month_date"].dt.year
df["sale_month"] = df["month_date"].dt.month
df["month_period"] = df["month_date"].dt.to_period("M").astype(str)

# Calculate continuous time feature: days since the very first transaction entry
df["days_since_start"] = (df["month_date"] - df["month_date"].min()).dt.days

df["flat_age_years"] = pd.to_numeric(df["sale_year"], errors="coerce") - pd.to_numeric(df["lease_commence_date"], errors="coerce")
df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease_years)
df["remaining_lease_months"] = (df["remaining_lease_years"] * 12).round()
df["remaining_lease_float"] = df["remaining_lease_years"]
df["mature_estate"] = df["town"].astype(str).str.strip().str.upper().map(lambda x: "Mature" if x in MATURE_ESTATES else "Immature")
df["region"] = df["town"].apply(classify_region)
df["region_URA"] = df["town"].apply(classify_ura_region)

for column in ["LATITUDE", "LONGITUDE", "mrt_name", "mrt_straight_distance", "mrt_walking_distance", "dist_to_cbd_m", "dist_nearest_park_m", "dist_nearest_hawker_centre_m", "sora_monthly_avg", "Per_Capita_GDP"]:
    if column not in df.columns:
        df[column] = np.nan

print("Missing region_URA mappings:", int(df["region_URA"].isna().sum()))
df[["town", "region", "region_URA", "mature_estate", "remaining_lease_years", "days_since_start"]].head()

Missing region_URA mappings: 0


,town,region,region_URA,mature_estate,remaining_lease_years,days_since_start
0,ANG MO KIO,OCR,North-East,Mature,70.0,0
1,ANG MO KIO,OCR,North-East,Mature,65.0,0
2,ANG MO KIO,OCR,North-East,Mature,64.0,0
3,ANG MO KIO,OCR,North-East,Mature,63.0,0
4,ANG MO KIO,OCR,North-East,Mature,64.0,0


## Backfill Cached MRT And Geospatial Features When Available

In [7]:
df_match = prepare_match_frame(df)
backfilled = False

for candidate in CACHED_FEATURE_CANDIDATES:
    if not candidate.exists():
        continue
    snap = load_snapshot(candidate)
    snap_match = prepare_match_frame(snap)
    available_backfill_cols = [column for column in BACKFILL_COLS if column in snap_match.columns]
    merged = df_match.merge(
        snap_match[MATCH_COLS + ["__match_occurrence"] + available_backfill_cols],
        on=MATCH_COLS + ["__match_occurrence"],
        how="left",
        suffixes=("", "__snap"),
    )
    matched_rows = 0
    for column in available_backfill_cols:
        snap_column = f"{column}__snap"
        if snap_column not in merged.columns:
            continue
        before = df[column].notna().sum()
        df[column] = df[column].combine_first(merged[snap_column])
        after = df[column].notna().sum()
        matched_rows = max(matched_rows, after - before)
    print(f"Backfill attempt from {candidate.name}: +{matched_rows:,} rows for at least one feature")
    backfilled = True

if not backfilled:
    print("No cached feature-added snapshots were found. Local-only derivations will still run.")

Backfill attempt from hdb_resale_with_new_features.csv: +0 rows for at least one feature
Backfill attempt from hdb_resale_with_walking_distance_FINAL.csv: +0 rows for at least one feature
Backfill attempt from merged_resale_dataset.csv: +0 rows for at least one feature


## Compute Local Enrichments And Optional External Merges

In [8]:
coord_mask = df["LATITUDE"].notna() & df["LONGITUDE"].notna()
df.loc[coord_mask, "dist_to_cbd_m"] = df.loc[coord_mask, "dist_to_cbd_m"].combine_first(
    haversine_distance(df.loc[coord_mask, "LATITUDE"], df.loc[coord_mask, "LONGITUDE"], 1.2841, 103.8515) * 1000
)
print("dist_to_cbd_m available rows:", int(df["dist_to_cbd_m"].notna().sum()))

if SORA_PATH.exists():
    sora_raw = pd.read_csv(
        SORA_PATH,
        skiprows=6,
        header=0,
        names=["year", "month", "day", "pub_date", "sora", "sora_index", "compound_1m", "compound_3m", "compound_6m"],
    )
    sora_raw = sora_raw[pd.to_numeric(sora_raw["sora"], errors="coerce").notna()].copy()
    sora_raw["year"] = sora_raw["year"].ffill()
    sora_raw["month"] = sora_raw["month"].ffill()
    sora_raw["date"] = pd.to_datetime(
        sora_raw["year"].astype(int).astype(str) + "-" + sora_raw["month"].astype(str) + "-" + sora_raw["day"].astype(int).astype(str),
        format="%Y-%b-%d",
    )
    sora_raw["sora"] = pd.to_numeric(sora_raw["sora"])
    sora_monthly = sora_raw.groupby(["year", "month"]).agg(sora_monthly_avg=("sora", "mean")).reset_index()
    sora_monthly["sale_year"] = sora_monthly["year"].astype(int)
    sora_monthly["sale_month"] = pd.to_datetime(sora_monthly["month"], format="%b").dt.month
    df = df.drop(columns=["sora_monthly_avg"], errors="ignore").merge(
        sora_monthly[["sale_year", "sale_month", "sora_monthly_avg"]],
        on=["sale_year", "sale_month"],
        how="left",
    )
    print("Merged monthly SORA values.")
else:
    print("Domestic_Interest_Rates.csv missing; keeping existing or missing SORA values.")

if coord_mask.any() and PARK_GEOJSON_PATH.exists() and HAWKER_GEOJSON_PATH.exists():
    try:
        from shapely.geometry import shape

        def load_centroids(path):
            geojson = json.loads(path.read_text())
            rows = []
            for feature in geojson["features"]:
                geom = shape(feature["geometry"])
                centroid = geom.centroid
                rows.append((centroid.y, centroid.x))
            return np.array(rows)

        parks = load_centroids(PARK_GEOJSON_PATH)
        hawkers = load_centroids(HAWKER_GEOJSON_PATH)

        def nearest_distance(lat, lon, points):
            distances = haversine_distance(lat, lon, points[:, 0], points[:, 1]) * 1000
            return float(np.min(distances))

        missing_park = coord_mask & df["dist_nearest_park_m"].isna()
        df.loc[missing_park, "dist_nearest_park_m"] = df.loc[missing_park].apply(lambda row: nearest_distance(row["LATITUDE"], row["LONGITUDE"], parks), axis=1)
        missing_hawker = coord_mask & df["dist_nearest_hawker_centre_m"].isna()
        df.loc[missing_hawker, "dist_nearest_hawker_centre_m"] = df.loc[missing_hawker].apply(lambda row: nearest_distance(row["LATITUDE"], row["LONGITUDE"], hawkers), axis=1)
        print("Filled park/hawker distances from GeoJSON files.")
    except ImportError:
        print("shapely not installed; park/hawker distances were not computed.")
else:
    print("GeoJSON files or coordinates missing; park/hawker distances were not computed.")

if GDP_PATH.exists():
    gdp_df = pd.read_csv(GDP_PATH)
    gdp_df["Year"] = pd.to_numeric(gdp_df["Year"], errors="coerce")
    df = df.drop(columns=["Per_Capita_GDP"], errors="ignore").merge(
        gdp_df[["Year", "Per_Capita_GDP"]],
        left_on="sale_year",
        right_on="Year",
        how="left",
    ).drop(columns=["Year"])
    print("Merged Per_Capita_GDP.")
else:
    print("GDP file missing; Per_Capita_GDP remains missing.")

dist_to_cbd_m available rows: 262449
Merged monthly SORA values.
shapely not installed; park/hawker distances were not computed.
Merged Per_Capita_GDP.


## Save The Final Modeling Dataset

In [9]:
date_mask = df["month_date"].between(PIPELINE_START_DATE, PIPELINE_END_DATE)
df = df.loc[date_mask].copy()
print(f"Applied pipeline date filter: {PIPELINE_START_DATE.date()} to {PIPELINE_END_DATE.date()}")
print(f"Filtered rows: {len(df):,}")
print(f"Filtered month range: {df['month'].min()} to {df['month'].max()}")
print(f"Rows still missing mrt_walking_distance: {int(df['mrt_walking_distance'].isna().sum()):,}")

final_df = build_final_dataset(df)

FINAL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(FINAL_OUTPUT_PATH, index=False)

print(f"Saved final modeling dataset to: {FINAL_OUTPUT_PATH}")
print(f"Final rows: {len(final_df):,}, columns: {len(final_df.columns)}")
print("Final dataset missing values:")
print(final_df.isna().sum())
final_df.head()


Applied pipeline date filter: 2015-01-01 to 2025-12-01
Filtered rows: 259,227
Filtered month range: 2015-01 to 2025-12
Rows still missing mrt_walking_distance: 578
Saved final modeling dataset to: C:\Users\Study\OneDrive - National University of Singapore\Desktop\Y3S2\CS3244\CS3244_group-3_ResalePriceModeling\data\hdb_resale_final_modeling_dataset_2015_2025.csv
Final rows: 259,227, columns: 19
Final dataset missing values:
town                              0
region                            0
region_URA                        0
mature_estate                     0
flat_type                         0
flat_model_raw                    0
flat_model                        0
floor_area_sqm                    0
storey_mid                        0
remaining_lease_years             0
mrt_walking_distance            578
dist_to_cbd_m                   402
dist_nearest_park_m             402
dist_nearest_hawker_centre_m    402
sora_monthly_avg                  0
Per_Capita_GDP                   

,town,region,region_URA,mature_estate,flat_type,flat_model_raw,flat_model,floor_area_sqm,storey_mid,remaining_lease_years,mrt_walking_distance,dist_to_cbd_m,dist_nearest_park_m,dist_nearest_hawker_centre_m,sora_monthly_avg,Per_Capita_GDP,days_since_start,resale_price,log_resale_price
0,ANG MO KIO,OCR,North-East,Mature,3 ROOM,Improved,Standard,60.0,8.0,70.0,371.0,10235.442419,339.862835,184.320761,0.143986,76503.0,0,255000.0,12.449019
1,ANG MO KIO,OCR,North-East,Mature,3 ROOM,New Generation,New Gen,68.0,2.0,65.0,1145.0,9998.296732,675.304013,181.896165,0.143986,76503.0,0,275000.0,12.524526
2,ANG MO KIO,OCR,North-East,Mature,3 ROOM,New Generation,New Gen,69.0,2.0,64.0,237.0,10055.896501,177.650627,155.570988,0.143986,76503.0,0,285000.0,12.560244
3,ANG MO KIO,OCR,North-East,Mature,3 ROOM,New Generation,New Gen,68.0,2.0,63.0,929.0,9312.549800,632.873927,124.093900,0.143986,76503.0,0,290000.0,12.577636
4,ANG MO KIO,OCR,North-East,Mature,3 ROOM,New Generation,New Gen,68.0,8.0,64.0,1336.0,9757.072331,813.589562,386.061555,0.143986,76503.0,0,290000.0,12.577636
